<a href="https://colab.research.google.com/github/muhammadwalad15-web/task/blob/main/Week3_10_tool_use_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [1]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [2]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '11:00:48', 'date': '2026-10-26', 'day': 'Monday'}


## Part 2: Tool Registry

Manage available tools.

In [3]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

#Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [4]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        # Simplified: pattern matching for demo
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            # Extract math expression from query
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer
        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer

# Test the agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))
print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Agent Tests:
Q: What is 25 * 4?
A: Error: invalid syntax (<string>, line 0)

Q: What time is it?
A: It's 11:01:02 on 2026-10-26 (Monday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [5]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 11:01:27 on 2026-10-26 (Monday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [ ]:
# YOUR CODE HERE

def my_custom_tool(arg: str) -> Dict[str, Any]:
    '''Your tool description.'''
    # Implement your tool
    return {'result': 'your result'}

# Add to twin
# twin.add_tool('my_tool', my_custom_tool, 'Description')
# Test: twin.chat('query that uses your tool')

In [6]:
import re

# --- 1. Define Memory and Task Storage ---
tasks_list = []
memory_db = {
    'hobbies': 'coding, playing guitar, and reading science fiction.',
    'favorite food': 'sushi and spicy ramen.',
    'favorite programming language': 'Python, because of its simplicity and AI ecosystem.'
}

# --- 2. Define Custom Tools ---
def task_manager(action_and_task: str) -> Dict[str, Any]:
    '''Manage a simple checklist. Usage: "add: clean room" or "list"'''
    action_and_task = action_and_task.strip()
    if action_and_task.lower().startswith('add:'):
        task_content = action_and_task[4:].strip()
        tasks_list.append(task_content)
        return {'result': f"Added task: '{task_content}'"}
    elif action_and_task.lower() == 'list':
        if not tasks_list:
            return {'result': 'Your task list is currently empty!'}
        formatted_tasks = [f"{i+1}. {task}" for i, task in enumerate(tasks_list)]
        return {'result': '\n'.join(formatted_tasks)}
    else:
        return {'error': 'Invalid action. Please use "add: [task]" or "list".'}

def memory_search(query: str) -> Dict[str, Any]:
    '''Search your digital twin\'s memories for details about your preferences.'''
    query_lower = query.lower()
    for key, value in memory_db.items():
        if key in query_lower:
            return {'result': f"I recall that your {key} is: {value}"}
    return {'result': "I couldn't find any specific memories matching that topic."}

# --- 3. Extend the Agent to recognize these new tools ---
# We subclass SimpleReActAgent to support our two new action routes
class AdvancedReActAgent(SimpleReActAgent):
    def think(self, query: str) -> str:
        query_lower = query.lower()
        if any(word in query_lower for word in ['task', 'to-do', 'todo', 'checklist']):
            return 'use_task_manager'
        elif any(word in query_lower for word in ['hobby', 'hobbies', 'favorite', 'remember', 'recall']):
            return 'use_memory_search'
        return super().think(query)

    def act(self, action: str, query: str) -> Dict[str, Any]:
        if action == 'use_task_manager':
            # Simple heuristic to extract argument for task manager
            query_lower = query.lower()
            if 'list' in query_lower:
                return self.registry.get_tool('task_manager')('list')
            match = re.search(r'(?:add\s+)?task\s+(.+)', query, re.IGNORECASE)
            if match:
                return self.registry.get_tool('task_manager')(f"add: {match.group(1).strip()}")
            return self.registry.get_tool('task_manager')('list')

        elif action == 'use_memory_search':
            return self.registry.get_tool('memory_search')(query)

        return super().act(action, query)

# --- 4. Subclass DigitalTwin to integrate our new agent type ---
class AdvancedDigitalTwin(DigitalTwinWithTools):
    def setup_tools(self):
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.registry.register('task_manager', task_manager, 'Manage tasks (add task [task_name] / list tasks)')
        self.registry.register('memory_search', memory_search, 'Search memories about hobbies or favorites')
        self.agent = AdvancedReActAgent(self.registry)

# --- 5. Instantiate and Test ---
my_twin = AdvancedDigitalTwin('Helper Twin')

print("--- Testing Task Manager Tool ---")
print(my_twin.chat('Add task Complete Week 3.1 Exercise'))
print(my_twin.chat('List my tasks'))

print("\n--- Testing Memory Search Tool ---")
print(my_twin.chat('What are my hobbies?'))
print(my_twin.chat('Do you remember my favorite programming language?'))

--- Testing Task Manager Tool ---
The result is Added task: 'Complete Week 3.1 Exercise'
The result is 1. Complete Week 3.1 Exercise

--- Testing Memory Search Tool ---
The result is I recall that your hobbies is: coding, playing guitar, and reading science fiction.
The result is I recall that your favorite programming language is: Python, because of its simplicity and AI ecosystem.


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI